<a href="https://colab.research.google.com/github/MusaR10/AAI2025/blob/2026fall/Coding_Exercise_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI

**Goal:** Simulate a customer service flow by chaining several prompts together. The output of each step becomes the input to the next.

**Chain:**
1. **Classify** the customer's issue (category + urgency)
2. **Extract** key details (product, problem, what the customer wants, sentiment)
3. **Draft** a support reply using the classification and details
4. **Review** the draft for tone and completeness, then output the final reply

**Tools:** Google Colab, Claude Opus 5.5, ChatGPT Luna 5.6, Open AIPython, Gemini API key



In [30]:
import google.generativeai as genai
from google.colab import userdata, files
from IPython.display import display
from PIL import Image as PILImage
import time
import os
import json


# Connect to Gemini
genai.configure(api_key=userdata.get("Prompt_Engineering_Key"))

model = genai.GenerativeModel("gemini-3.1-flash-lite")

print("Gemini initialized successfully.")



Gemini initialized successfully.


In [31]:
def ask(prompt, temperature=0.3, as_json=False):
    """Send one prompt to Gemini and return the text (or parsed JSON)."""
    config = {"temperature": temperature}
    if as_json:
        config["response_mime_type"] = "application/json"

    response = model.generate_content(prompt, generation_config=config)
    time.sleep(3)  # small pause to stay under free-tier rate limits

    text = response.text.strip()
    if as_json:
        text = text.replace("```json", "").replace("```", "").strip()
        return json.loads(text)
    return text


def show(label, data):
    """Print one step's output with a label."""
    print(f"\n--- {label} ---")
    print(json.dumps(data, indent=2) if isinstance(data, (dict, list)) else data)

## Step 1: Classify the issue

In [32]:
def classify_issue(customer_message):
    prompt = f"""You are a customer support triage assistant for an online store called ShopEase.
Classify the customer message below.

Return JSON with exactly these keys:
- "category": one of ["billing", "shipping", "product_defect", "safety", "account", "other"]
- "urgency": one of ["low", "medium", "high"]
- "sentiment": one of ["positive", "neutral", "frustrated", "angry"]
- "reason": one short sentence explaining your choice

Rules:
- Use "safety" if the message mentions smoke, fire, melting, shock, or injury.
- Urgency is "high" for safety issues or money charged in error.

Customer message:
\"\"\"{customer_message}\"\"\""""
    return ask(prompt, temperature=0, as_json=True)

## Step 2: Gather missing info (uses Step 1 output)
The Step 1 category decides which details are required. If any are missing, the chain asks the customer and re-checks.


In [33]:
def gather_missing_info(conversation, classification):
    prompt = f"""You are a support assistant checking whether a ticket has enough information to be resolved.

Step 1 classified this ticket as: {json.dumps(classification)}

Required information for each category:
- product_defect: order number, product name, description of the damage
- shipping: order number, description of the delivery problem
- billing: date of the charge, amount charged, last 4 digits of the card
- safety: order number, product name, whether anyone was injured
- account: the email address currently on the account
- other: a clear description of the request

Using the category from Step 1, check the conversation for each required field.

Return JSON with exactly these keys:
- "known_info": an object of the required fields that ARE present, with their values
- "missing_info": a list of required fields that are NOT present
- "clarifying_questions": one short, polite question per missing field (empty list if none)

Rules:
- Only mark a field as known if it is explicitly stated. Never invent order numbers or amounts.
- Never ask for full card numbers, passwords, or security codes.

Conversation:
\"\"\"{conversation}\"\"\""""
    return ask(prompt, temperature=0, as_json=True)


## Step 3: Propose a solution (uses Steps 1 and 2)

In [34]:
POLICY = """
- Damaged or defective items: free replacement or full refund within 30 days of delivery.
- Duplicate or incorrect charges: refunded to the original payment method within 5-7 business days.
- Account changes: the customer updates their email in Settings > Account; support can send a reset link.
- Safety issues (smoke, fire, shock, injury): customer must stop using the product; full refund.
- Never ask for full card numbers, passwords, or security codes.
- Never promise anything not listed in this policy.
"""

In [35]:
def propose_solution(classification, info):
    prompt = f"""You are a senior support agent who resolves tickets according to company policy.

Step 1 classification: {json.dumps(classification)}
Step 2 gathered information: {json.dumps(info)}



Company policy:
{POLICY}

Propose a resolution that follows the policy exactly.

Return JSON with exactly these keys:
- "proposed_solution": one sentence describing the resolution
- "next_steps": a list of 2 to 4 concrete actions
- "confidence": one of ["high", "medium", "low"] (use "low" if the policy does not clearly cover this case)

Rules:
- Do not offer discounts, gift cards, or anything not in the policy.
- If Step 2 still lists missing information, the first next step must be collecting it."""
    return ask(prompt, temperature=0.2, as_json=True)

## Step 4: Escalation rule (uses Steps 1, 2, and 3)

In [36]:
def check_escalation(classification, info, solution):
    prompt = f"""You are an escalation checker for a customer support team.

Step 1 classification: {json.dumps(classification)}
Step 2 gathered information: {json.dumps(info)}
Step 3 proposed solution: {json.dumps(solution)}

Escalate to a human specialist if ANY of these rules is true:
1. The category is "safety" or anyone was injured.
2. A refund or charge amount is over $100.
3. Sentiment is "angry" AND urgency is "high".
4. Step 3 confidence is "low".
5. The customer mentions a lawyer or asks for a manager.

Return JSON with exactly these keys:
- "escalate": true or false
- "rules_triggered": a list of the rule numbers that apply (empty list if none)
- "escalate_to": one of ["none", "billing_team", "product_safety_team", "supervisor"]
- "reason": one sentence

Apply the rules literally. Do not escalate for any reason not listed."""
    return ask(prompt, temperature=0, as_json=True)

## Step 5: Final reply (uses Steps 1–4)

In [37]:
def write_reply(conversation, classification, info, solution, escalation):
    prompt = f"""You are a friendly, professional customer support agent for ShopEase.

Step 1 classification: {json.dumps(classification)}
Step 2 gathered information: {json.dumps(info)}
Step 3 proposed solution: {json.dumps(solution)}
Step 4 escalation decision: {json.dumps(escalation)}

Write the final reply to the customer.

Rules:
- If sentiment is frustrated or angry, open with one sincere sentence of apology.
- Mention the order number or product if known.
- Explain the solution and next steps in plain language.
- If Step 2 still lists missing information, ask its clarifying questions.
- If escalate is true, say a specialist team will contact them within 1 business day.
- If it is a safety issue, tell them to stop using the product.
- Do not promise anything that is not in the proposed solution.
- Keep it under 150 words. Sign off as "The ShopEase Support Team".

Conversation:
\"\"\"{conversation}\"\"\""""
    return ask(prompt, temperature=0.4)

In [38]:
def support_chain(customer_message, customer_followup=None):
    print("=" * 70)
    print("CUSTOMER MESSAGE:\n" + customer_message)
    conversation = f"Customer: {customer_message}"

    # Step 1
    classification = classify_issue(customer_message)
    show("STEP 1 - CLASSIFICATION", classification)

    # Step 2 (uses Step 1)
    info = gather_missing_info(conversation, classification)
    show("STEP 2 - GATHER MISSING INFO", info)

    # If info is missing, ask the customer, add their answer, and re-check
    if info["missing_info"] and customer_followup:
        questions = " ".join(info["clarifying_questions"])
        print("\n>> Asking the customer:", questions)
        print(">> Customer replies (simulated):", customer_followup)
        conversation += f"\nSupport: {questions}\nCustomer: {customer_followup}"
        info = gather_missing_info(conversation, classification)
        show("STEP 2 (RE-CHECK AFTER CUSTOMER'S ANSWER)", info)

    # Step 3 (uses Steps 1-2)
    solution = propose_solution(classification, info)
    print("STEP 3 - PROPOSED SOLUTION", solution)

    # Step 4 (uses Steps 1-3)
    escalation = check_escalation(classification, info, solution)
    show("STEP 4 - ESCALATION DECISION", escalation)

    # Step 5 (uses Steps 1-4)
    reply = write_reply(conversation, classification, info, solution, escalation)
    show("STEP 5 - FINAL REPLY", reply)
    return reply

#TEST CASES


In [40]:
test_cases = [
    # Complete info, frustrated customer -> replacement, likely no escalation
    ("I ordered a blender two weeks ago (order #48213) and it arrived with a cracked jar. "
     "This is really frustrating because it was a birthday gift. I want a replacement ASAP.",
     None),

    # Missing info -> Step 2 asks for it -> amount over $100 triggers escalation
    ("Hi, I was charged twice for my order this month. Can you refund the extra charge?",
     "It was on September 3rd, $249.99 each time, card ending in 4417."),

    # Safety issue -> must escalate to the product safety team
    ("The space heater I bought last week (order #55120) started smoking and melted its plug. "
     "My kid was sitting right next to it!",
     "No one was hurt, thankfully."),

    #A lot of missing info and unrealistic ask
    ("I orderd a Logitech G502 Hero mouse, but it arrived broken, please fix this and I should be compensated a free gaming laptop at least.",
     None)

]

for message, followup in test_cases:
    support_chain(message, followup)

CUSTOMER MESSAGE:
I ordered a blender two weeks ago (order #48213) and it arrived with a cracked jar. This is really frustrating because it was a birthday gift. I want a replacement ASAP.

--- STEP 1 - CLASSIFICATION ---
{
  "category": "product_defect",
  "urgency": "medium",
  "sentiment": "frustrated",
  "reason": "The customer received a damaged item and is requesting a replacement."
}

--- STEP 2 - GATHER MISSING INFO ---
{
  "known_info": {
    "order number": "48213",
    "product name": "blender",
    "description of the damage": "cracked jar"
  },
  "missing_info": [],
  "clarifying_questions": []
}


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1472.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 735.01ms


STEP 3 - PROPOSED SOLUTION {'proposed_solution': 'We will provide a free replacement for your damaged blender jar as per our company policy for defective items.', 'next_steps': ['Verify the delivery date of order 48213 to ensure it falls within the 30-day replacement window.', 'Process the shipment of a replacement blender jar to the original shipping address on file.', 'Provide the customer with a new tracking number once the replacement has been dispatched.'], 'confidence': 'high'}

--- STEP 4 - ESCALATION DECISION ---
{
  "escalate": false,
  "rules_triggered": [],
  "escalate_to": "none",
  "reason": "The request does not meet any of the specified criteria for escalation to a human specialist."
}

--- STEP 5 - FINAL REPLY ---
Dear customer, please accept our sincere apologies for the frustration caused by receiving your blender with a cracked jar, especially since it was intended as a birthday gift.

We would like to make this right for you. As per our company policy for defective 

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 863.15ms



--- STEP 4 - ESCALATION DECISION ---
{
  "escalate": true,
  "rules_triggered": [
    2
  ],
  "escalate_to": "billing_team",
  "reason": "The refund amount of $249.99 exceeds the $100 threshold defined in the escalation rules."
}

--- STEP 5 - FINAL REPLY ---
Hello,

Thank you for reaching out to us regarding the duplicate charge on your account. I have reviewed your report for the charge of $249.99 on September 3rd using your card ending in 4417.

Because this refund request exceeds our standard automated limit, I have escalated your case to our billing team for further review. A specialist will contact you within one business day to finalize the process. Once approved, the refund will be credited back to your original payment method within 5-7 business days.

We appreciate your patience while we resolve this for you.

The ShopEase Support Team
CUSTOMER MESSAGE:
The space heater I bought last week (order #55120) started smoking and melted its plug. My kid was sitting right next to i

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 939.05ms



--- STEP 1 - CLASSIFICATION ---
{
  "category": "safety",
  "urgency": "high",
  "sentiment": "angry",
  "reason": "The customer reported a fire hazard involving a melting product that posed a direct risk to their child."
}

--- STEP 2 - GATHER MISSING INFO ---
{
  "known_info": {
    "order number": "55120",
    "product name": "space heater",
    "whether anyone was injured": "no"
  },
  "missing_info": [],
  "clarifying_questions": []
}
STEP 3 - PROPOSED SOLUTION {'proposed_solution': 'We will issue a full refund for your space heater immediately due to the safety concern reported.', 'next_steps': ['Instruct the customer to immediately stop using the space heater and unplug it to prevent further risk.', 'Process a full refund for order 55120 to the original payment method.', 'Escalate this report to our quality assurance team for a safety investigation.'], 'confidence': 'high'}

--- STEP 4 - ESCALATION DECISION ---
{
  "escalate": true,
  "rules_triggered": [
    1,
    3
  ],
  "e